In [6]:
import torch
from torch import nn
import ptwt
import numpy as np
from torch_geometric.nn.conv import GATConv
from torch_geometric.nn.norm import GraphNorm
from torch_geometric.nn import Sequential
from torch_geometric.data import Data, Batch
from dataset import EEGDataset
from torch_geometric.loader import DataLoader
import torch.utils.data as torchdata
import tqdm
from math import floor
import gc
from torchaudio.transforms import PSD

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [3]:
train_frac = 0.02
val_frac = 0.001

dataset = EEGDataset(root="/home/hice1/mchen439/data/TUH-Processed", frac=train_frac + val_frac)
num_train = int(len(dataset) * (train_frac / (train_frac + val_frac)))
num_val = len(dataset) - num_train
train_dataset, val_dataset = torchdata.random_split(dataset, [num_train, num_val])
print("Train and Validation Dataset Length: ", len(train_dataset), len(val_dataset))

Loading 314 subjects


100%|██████████| 314/314 [00:19<00:00, 16.47it/s]


Train and Validation Dataset Length:  14524 727


In [6]:
train_loader = DataLoader(train_dataset, batch_size=1024, num_workers=0, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1024, num_workers=0, shuffle=False)
print(len(train_loader), len(val_loader))

15 1


In [2]:
BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma']

In [131]:
class WaveletEncoderDecoder(nn.Module):
    def __init__(self, num_channels, conv_kernel_size, conv_kernel_stride, seq_len, heads, encoded_h, decoder_size, decoder_stride):
        super(WaveletEncoderDecoder, self).__init__()

        assert len(decoder_size) == len(decoder_stride)

        self.num_channels = num_channels
        self.conv_kernel_size = conv_kernel_size
        self.conv_kernel_stride = conv_kernel_stride
        self.seq_len = seq_len
        self.heads = heads
        self.encoded_h = encoded_h

        self.patch_embedder = nn.Sequential(
            nn.Conv1d(self.num_channels, self.num_channels, 
            self.conv_kernel_size, stride=self.conv_kernel_stride,
            padding=self.conv_kernel_size//2),
            nn.Dropout1d(0.1),
            nn.GroupNorm(1, self.num_channels), # Same as Layer Norm
            nn.GELU()
        )

        L_out = self.seq_len + 2 * (self.conv_kernel_size//2) - 1 * (self.conv_kernel_size - 1) - 1
        L_out = floor(L_out / self.conv_kernel_stride) + 1
        print("L_out: ", L_out)
        self.gnn_embedder = GATConv(L_out, self.encoded_h, heads=self.heads)
        self.gnn_group_norm = nn.GroupNorm(1, self.encoded_h * self.heads)
        self.gnn_dropout = nn.Dropout(p=0.1)
        self.gnn_gelu = nn.GELU()
        L_out = self.encoded_h * self.heads
        self.decoders = nn.Sequential()
        for i, (w, s) in enumerate(zip(decoder_size, decoder_stride)):
            if i == len(decoder_size) - 1:
                output_padding = self.seq_len - L_out
                self.decoders.add_module("Decoder_Final".format(i),
                nn.Sequential(
                    nn.ConvTranspose1d(self.num_channels, self.num_channels,
                    w, stride=s, padding=w//2, output_padding=0 if output_padding < 0 else output_padding),
                    nn.Dropout(0.1),
                    nn.GroupNorm(1, self.num_channels),
                    nn.GELU()
                    )
                )
                if output_padding < 0:
                    L_out = (L_out - 1) * s - 2 * (w//2) + 1 * (w - 1) + 1
                    print(L_out, self.seq_len)
                    self.decoders.add_module("Decoder_Final_Linear".format(i), nn.Linear(L_out, self.seq_len))
            else:
                self.decoders.add_module("Decoder_{}".format(i),
                nn.Sequential(
                    nn.ConvTranspose1d(self.num_channels, self.num_channels,
                    w, stride=s, padding=w//2),
                    nn.Dropout(0.1),
                    nn.GroupNorm(1, self.num_channels),
                    nn.GELU()
                    )
                )
                L_out = (L_out - 1) * s - 2 * (w//2) + 1 * (w - 1) + 1
    
    def forward(self, graph, x):
        patch_embedding = self.patch_embedder(x)

        graph_x = graph.clone()

        edge_index = graph_x.edge_index.to(device)
        edge_dist = graph.edge_attr.to(device)

        patch_embedding = patch_embedding.view(-1, patch_embedding.shape[-1])
        
        encoding = self.gnn_embedder(patch_embedding, edge_index, edge_dist)
        encoding = encoding.view(-1, self.num_channels, self.encoded_h * self.heads)
        decoding = self.decoders(encoding)

        return encoding, decoding 


In [132]:
class EncoderDecoder(nn.Module):
    def __init__(self):
        super(EncoderDecoder, self).__init__()

        self.encoder_decoders = nn.ParameterDict({
            'delta': WaveletEncoderDecoder(
                    num_channels = 19,
                    conv_kernel_size = 2,
                    conv_kernel_stride = 2,
                    seq_len = 246,
                    heads = 4,
                    encoded_h = 120,
                    decoder_size = (2, 2, 1),
                    decoder_stride = (2, 2, 1)
                    ),

            'theta': WaveletEncoderDecoder(
                    num_channels = 19,
                    conv_kernel_size = 2,
                    conv_kernel_stride = 2,
                    seq_len = 246,
                    heads = 4,
                    encoded_h = 120,
                    decoder_size = (2, 2, 1),
                    decoder_stride = (2, 2, 1)
                    ),

            'alpha': WaveletEncoderDecoder(
                    num_channels = 19,
                    conv_kernel_size = 2,
                    conv_kernel_stride = 2,
                    seq_len = 486,
                    heads = 4,
                    encoded_h = 240,
                    decoder_size = (2, 2, 1),
                    decoder_stride = (2, 2, 1)
                    ),

            'beta': WaveletEncoderDecoder(
                    num_channels = 19,
                    conv_kernel_size = 2,
                    conv_kernel_stride = 2,
                    seq_len = 966,
                    heads = 4,
                    encoded_h = 480,
                    decoder_size = (2, 2, 1),
                    decoder_stride = (2, 2, 1)
                    ),

            'gamma': WaveletEncoderDecoder(
                    num_channels = 19,
                    conv_kernel_size = 2,
                    conv_kernel_stride = 2,
                    seq_len = 1926,
                    heads = 4,
                    encoded_h = 960,
                    decoder_size = (2, 2, 1),
                    decoder_stride = (2, 2, 1)
                    ),
        })

    def forward(self, graph, data):
        assert data.keys() == self.encoder_decoders.keys()
        output = {}
        for band, band_decomposition in data.items():
            output[band] = self.encoder_decoders[band](graph, band_decomposition)
        return output
    

In [133]:
model = EncoderDecoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()
print("Total parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

L_out:  124
1914 246
L_out:  124
1914 246
L_out:  244
3834 486
L_out:  484
7674 966
L_out:  964
15354 1926
Total parameters: 44830265


In [143]:
test_data = torch.randn(4, 19, 15360).to(device)
edge_indices = []
edge_index = torch.tensor([[0, 1, 2, 3], [1, 2, 3, 0]]).to(device)
for idx in range(test_data.shape[0]):
	edge_index_copy = edge_index.clone()
	edge_index_copy += idx * 19
	edge_indices.append(edge_index_copy)
batch_edge_index = torch.cat(edge_indices, dim=1)
print(batch_edge_index.shape)

edge_attributes = []
for idx in range(test_data.shape[0]):
	edge_attributes.append(torch.randn(4, 1).clone())
batch_edge_attributes = torch.cat(edge_attributes, dim=0)
print(batch_edge_attributes.shape)

test_graph = Data(x=test_data, edge_index=batch_edge_index, edge_attr=batch_edge_attributes.to(device))

torch.Size([2, 16])
torch.Size([16, 1])


In [141]:
dbt_dwpt = ptwt.WaveletPacket(test_data,
								wavelet='db4',
								mode='reflect',
								maxlevel=6,
								axis=-1)
relevant_bands = [dbt_dwpt['aaaaaa'], dbt_dwpt['aaaaad'], dbt_dwpt['aaaad'],
		 dbt_dwpt['aaad'], dbt_dwpt['aad']]

for band in relevant_bands:
	print(band.shape)

inputs = dict(zip(BANDS, relevant_bands))

torch.Size([4, 19, 246])
torch.Size([4, 19, 246])
torch.Size([4, 19, 486])
torch.Size([4, 19, 966])
torch.Size([4, 19, 1926])


In [144]:
output = model(test_graph, inputs)

torch.Size([4, 19, 246])
Patch Embedding Shape:  torch.Size([76, 124])
Edge Index Shape:  torch.Size([2, 16])
Edge Dist Shape:  torch.Size([16, 1])
Encoding Shape:  torch.Size([76, 480])
Encoding Reshaped Shape:  torch.Size([4, 19, 480])
torch.Size([4, 19, 246])
Patch Embedding Shape:  torch.Size([76, 124])
Edge Index Shape:  torch.Size([2, 16])
Edge Dist Shape:  torch.Size([16, 1])
Encoding Shape:  torch.Size([76, 480])
Encoding Reshaped Shape:  torch.Size([4, 19, 480])


KeyboardInterrupt: 

In [26]:
def train_one_epoch(model, optimizer, loss_fn, train_loader):
	loss_sum = 0
	model.train()

	pbar = tqdm.trange(len(train_loader), desc="Training")
	data_iterator = iter(train_loader)
	for iteration in pbar:
		graph = next(data_iterator)
		x = torch.tensor(np.array(graph.x)).float().to(device)
		# data = _std_norm(data)
		dbt_dwpt = ptwt.WaveletPacket(x,
									wavelet='db4',
									mode='reflect',
									maxlevel=6,
									axis=-1)
		relevant_bands = [dbt_dwpt['aaaaaa'], dbt_dwpt['aaaaad'], dbt_dwpt['aaaad'],
		 dbt_dwpt['aaad'], dbt_dwpt['aad']]

		inputs = dict(zip(BANDS, relevant_bands))

		optimizer.zero_grad()

		encodings, decodings = model(graph, inputs)

		delta_loss = loss_fn(inputs['delta'], decodings['delta']) + fft_loss(inputs['delta'], decodings['delta'])
		theta_loss = loss_fn(inputs['theta'], decodings['theta']) + fft_loss(inputs['theta'], decodings['theta'])
		alpha_loss = loss_fn(inputs['alpha'], decodings['alpha']) + fft_loss(inputs['alpha'], decodings['alpha'])
		beta_loss =  loss_fn(inputs['beta'], decodings['beta']) + fft_loss(inputs['beta'], decodings['beta'])
		gamma_loss = loss_fn(inputs['gamma'], decodings['gamma']) + fft_loss(inputs['gamma'], decodings['gamma'])

		loss = delta_loss + theta_loss + alpha_loss + beta_loss + gamma_loss
		loss.backward()
		optimizer.step()
		loss_sum += loss.item()

		pbar.set_postfix(loss=loss.item(),
			delta_loss=delta_loss.item(),
			theta_loss=theta_loss.item(),
			alpha_loss=alpha_loss.item(),
			beta_loss=beta_loss.item(),
			gamma_loss=gamma_loss.item())
	return loss_sum

In [27]:
def validate_one_epoch(model, loss_fn, val_loader):
	loss_sum = 0
	model.eval()

	pbar = tqdm.trange(len(val_loader), desc="Validating")
	data_iterator = iter(val_loader)
	for iteration in pbar:
		graph = next(data_iterator)
		x = torch.tensor(np.array(graph.x)).float().to(device)
		# data = _std_norm(data)
		dbt_dwpt = ptwt.WaveletPacket(x,
									wavelet='db4',
									mode='reflect',
									maxlevel=6,
									axis=-1)
		relevant_bands = [dbt_dwpt['aaaaaa'], dbt_dwpt['aaaaad'], dbt_dwpt['aaaad'],
		 dbt_dwpt['aaad'], dbt_dwpt['aad']]

		inputs = dict(zip(BANDS, relevant_bands))

		encodings, decodings = model(graph, inputs)

		delta_loss = loss_fn(inputs['delta'], decodings['delta']) + fft_loss(inputs['delta'], decodings['delta'])
		theta_loss = loss_fn(inputs['theta'], decodings['theta']) + fft_loss(inputs['theta'], decodings['theta'])
		alpha_loss = loss_fn(inputs['alpha'], decodings['alpha']) + fft_loss(inputs['alpha'], decodings['alpha'])
		beta_loss =  loss_fn(inputs['beta'], decodings['beta']) + fft_loss(inputs['beta'], decodings['beta'])
		gamma_loss = loss_fn(inputs['gamma'], decodings['gamma']) + fft_loss(inputs['gamma'], decodings['gamma'])

		loss = delta_loss + theta_loss + alpha_loss + beta_loss + gamma_loss
		loss_sum += loss.item()

		pbar.set_postfix(loss=loss.item(),
			delta_loss=delta_loss.item(),
			theta_loss=theta_loss.item(),
			alpha_loss=alpha_loss.item(),
			beta_loss=beta_loss.item(),
			gamma_loss=gamma_loss.item())

	return loss_sum

In [29]:
for epoch_idx in range(5):
	training_loss = train_one_epoch(model, optimizer, loss_fn, train_loader)
	val_loss = validate_one_epoch(model, loss_fn, val_loader)
	print("Epoch: ", epoch_idx, "Total Training Loss: ", training_loss, "Total Validation Loss: ", val_loss)

Training:   0%|          | 0/15 [01:25<?, ?it/s]


KeyboardInterrupt: 